In [0]:
bronze_base = "abfss://shopsphere@stshopsphere2026.dfs.core.windows.net/bronze"

customers_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/customers")
)

products_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/products")
)

orders_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_base}/orders")
)

In [0]:
customers_bronze.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [0]:
display(customers_bronze.limit(20))

customer_id,first_name,last_name,email,gender,date_of_birth,city,state,country,signup_date
CUST-1001,Patricia,Thomas,patricia.thomas45@outlook.com,F,1972-03-19,Houston,TX,USA,2022-11-07
CUST-1002,Nancy,Thomas,nancy.thomas37@outlook.com,F,1985-05-27,Los Angeles,CA,USA,2021-06-30
CUST-1003,Elizabeth,Hernandez,elizabeth.hernandez15@outlook.com,F,1994-07-03,New York,NY,USA,2023-08-05
CUST-1004,Mary,Miller,mary.miller67@hotmail.com,F,1993-01-20,Atlanta,GA,USA,2023-06-25
CUST-1005,Michael,Rodriguez,michael.rodriguez71@yahoo.com,M,1996-04-28,Miami,FL,USA,2023-09-20
CUST-1006,James,Martin,james.martin31@yahoo.com,M,1989-10-10,New York,NY,USA,2023-04-16
CUST-1007,Michael,Brown,michael.brown60@gmail.com,M,1987-05-11,Phoenix,AZ,USA,2022-08-21
CUST-1008,Susan,Thomas,susan.thomas61@yahoo.com,F,1982-03-17,Phoenix,AZ,USA,2022-01-26
CUST-1009,Mark,Williams,mark.williams36@outlook.com,M,1965-02-12,Dallas,TX,USA,2022-08-03
CUST-1010,Margaret,Jackson,margaret.jackson75@outlook.com,F,1989-02-28,Miami,FL,USA,2023-09-19


In [0]:
print("Rows:", customers_bronze.count())
print("Columns:", len(customers_bronze.columns))

Rows: 1027
Columns: 10


In [0]:
from pyspark.sql.functions import col, count

customer_duplicates = (
    customers_bronze
    .groupBy("customer_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

display(customer_duplicates)

customer_id,count
CUST-1077,2
CUST-1137,2
CUST-1175,2
CUST-1211,2
CUST-1281,2
CUST-1320,2
CUST-1372,2
CUST-1412,2
CUST-1514,2
CUST-1522,2


In [0]:
product_duplicates = (
    products_bronze
    .groupBy("product_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

display(product_duplicates)

product_id,count


In [0]:
display(
    products_bronze
    .filter(
        (col("price") <= 0) |
        (col("cost") < 0)
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-99999,GAP Discontinued Item,Men,T-Shirts,Men,Unknown,M,GAP,null,-10.0,-5


In [0]:
order_duplicates = (
    orders_bronze
    .groupBy("order_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

display(order_duplicates)

order_id,count
ORD-100034,2
ORD-100036,2
ORD-100040,2
ORD-100097,2
ORD-100108,2
ORD-100120,2
ORD-100252,2
ORD-100322,2
ORD-100440,2
ORD-100511,2


In [0]:
display(
    orders_bronze
    .filter(col("quantity") <= 0)
)

order_id,customer_id,product_id,order_date,quantity,unit_price,discount,shipping_cost,payment_method,order_status,total_amount
ORD-100007,CUST-1617,PRD-176,2022-08-16T03:25:00.000Z,-2,20.01,0.5,5.99,PayPal,Cancelled,-14.02
ORD-100029,CUST-1797,PRD-135,2022-05-30T07:17:00.000Z,-2,79.91,0.1,5.99,Credit Card,Completed,-137.85
ORD-100357,CUST-1221,PRD-177,2023-05-14T09:36:00.000Z,-2,23.9,0.0,5.99,GAP Gift Card,Completed,-41.81
ORD-100360,CUST-1451,PRD-115,2024-02-14T05:31:00.000Z,-3,109.54,0.1,0.0,GAP Gift Card,Completed,-295.76
ORD-100435,CUST-1553,PRD-188,2023-08-24T10:55:00.000Z,-3,46.88,0.5,5.99,Credit Card,Completed,-64.33
ORD-100511,CUST-1061,PRD-161,2022-09-10T07:40:00.000Z,-2,50.06,0.2,5.99,Apple Pay,Shipped,-74.11
ORD-100672,CUST-1789,PRD-171,2023-03-06T18:21:00.000Z,-1,52.26,0.4,5.99,GAP Gift Card,Completed,-25.37
ORD-101084,CUST-1549,PRD-164,2023-02-16T09:57:00.000Z,-3,50.6,0.15,0.0,Klarna,Completed,-129.03
ORD-101100,CUST-1741,PRD-148,2024-04-25T18:10:00.000Z,-3,20.71,0.5,5.99,Apple Pay,Completed,-25.08
ORD-101163,CUST-1039,PRD-125,2023-11-17T17:34:00.000Z,-2,14.2,0.0,0.0,GAP Gift Card,Processing,-28.4


In [0]:
display(
    orders_bronze
    .filter(
        (col("discount") < 0) |
        (col("discount") > 1)
    )
)

order_id,customer_id,product_id,order_date,quantity,unit_price,discount,shipping_cost,payment_method,order_status,total_amount
ORD-100042,CUST-1542,PRD-166,2024-07-09T04:26:00.000Z,4,49.85,2.2,0.0,Apple Pay,Completed,-239.28
ORD-100171,CUST-1223,PRD-137,2024-04-25T18:07:00.000Z,3,43.59,1.66,0.0,Credit Card,Completed,-86.31
ORD-100307,CUST-1448,PRD-112,2024-01-12T07:09:00.000Z,3,59.65,1.41,0.0,Credit Card,Cancelled,-73.37
ORD-100316,CUST-1184,PRD-155,2023-06-29T12:55:00.000Z,2,29.48,2.43,0.0,Klarna,Completed,-84.31
ORD-100365,CUST-1323,PRD-158,2022-11-12T13:36:00.000Z,4,55.89,2.42,0.0,Credit Card,Processing,-317.46
ORD-100405,CUST-1996,PRD-154,2024-07-04T15:29:00.000Z,3,24.23,1.53,0.0,Klarna,Completed,-38.53
ORD-100628,CUST-1836,PRD-142,2024-02-03T22:27:00.000Z,2,65.0,2.22,0.0,Apple Pay,Processing,-158.6
ORD-100635,CUST-1544,PRD-130,2022-02-11T07:52:00.000Z,4,44.2,1.26,0.0,Credit Card,Completed,-45.97
ORD-100660,CUST-1911,PRD-116,2023-12-25T13:16:00.000Z,4,127.61,1.35,0.0,Klarna,Completed,-178.65
ORD-100928,CUST-1474,PRD-110,2024-04-09T13:37:00.000Z,1,68.48,1.83,0.0,Credit Card,Returned,-56.84


In [0]:
display(
    orders_bronze
    .filter(
        col("customer_id").isNull() |
        (col("customer_id") == "")
    )
)

order_id,customer_id,product_id,order_date,quantity,unit_price,discount,shipping_cost,payment_method,order_status,total_amount
ORD-100051,null,PRD-126,2022-04-04T01:24:00.000Z,3,24.75,0.1,0.0,PayPal,Shipped,66.83
ORD-100093,null,PRD-146,2024-02-09T09:25:00.000Z,3,35.41,0.4,0.0,Apple Pay,Completed,63.74
ORD-100103,null,PRD-114,2022-10-25T12:10:00.000Z,1,115.66,0.5,0.0,GAP Gift Card,Completed,57.83
ORD-100121,null,PRD-154,2022-02-03T00:01:00.000Z,3,24.23,0.3,0.0,Klarna,Shipped,50.88
ORD-100125,null,PRD-199,2023-08-31T04:01:00.000Z,1,21.93,0.0,5.99,PayPal,Cancelled,27.92
ORD-100155,null,PRD-159,2022-07-11T08:01:00.000Z,4,35.8,0.0,0.0,Klarna,Completed,143.2
ORD-100257,null,PRD-193,2022-01-22T12:05:00.000Z,3,31.54,0.5,0.0,Klarna,Completed,47.31
ORD-100385,null,PRD-149,2024-04-02T13:25:00.000Z,1,19.19,0.15,5.99,GAP Gift Card,Completed,22.3
ORD-100453,null,PRD-131,2023-11-07T10:11:00.000Z,2,52.01,0.5,0.0,GAP Gift Card,Completed,52.01
ORD-100465,null,PRD-101,2023-07-31T01:52:00.000Z,1,24.45,0.5,5.99,Klarna,Returned,18.21


In [0]:
from pyspark.sql.functions import lower, trim

customers_clean = (
    customers_bronze
    .withColumn("email", lower(trim(col("email"))))
)

In [0]:
customers_clean = (
    customers_clean
    .dropDuplicates(["customer_id"])
)

In [0]:
from pyspark.sql.functions import col, trim, lower, when

customers_clean = (
    customers_bronze
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("gender", trim(col("gender")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("country", trim(col("country")))
)

In [0]:
customers_clean = customers_clean.filter(
    col("customer_id").isNotNull() &
    (trim(col("customer_id")) != "")
)

In [0]:
customers_clean = customers_clean.dropDuplicates(["customer_id"])

In [0]:
customers_clean = customers_clean.withColumn(
    "email",
    when(
        col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"),
        col("email")
    ).otherwise(None)
)

In [0]:
customers_clean.printSchema()
display(customers_clean.limit(20))

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



customer_id,first_name,last_name,email,gender,date_of_birth,city,state,country,signup_date
CUST-1001,Patricia,Thomas,patricia.thomas45@outlook.com,F,1972-03-19,Houston,TX,USA,2022-11-07
CUST-1002,Nancy,Thomas,nancy.thomas37@outlook.com,F,1985-05-27,Los Angeles,CA,USA,2021-06-30
CUST-1003,Elizabeth,Hernandez,elizabeth.hernandez15@outlook.com,F,1994-07-03,New York,NY,USA,2023-08-05
CUST-1004,Mary,Miller,mary.miller67@hotmail.com,F,1993-01-20,Atlanta,GA,USA,2023-06-25
CUST-1005,Michael,Rodriguez,michael.rodriguez71@yahoo.com,M,1996-04-28,Miami,FL,USA,2023-09-20
CUST-1006,James,Martin,james.martin31@yahoo.com,M,1989-10-10,New York,NY,USA,2023-04-16
CUST-1007,Michael,Brown,michael.brown60@gmail.com,M,1987-05-11,Phoenix,AZ,USA,2022-08-21
CUST-1008,Susan,Thomas,susan.thomas61@yahoo.com,F,1982-03-17,Phoenix,AZ,USA,2022-01-26
CUST-1009,Mark,Williams,mark.williams36@outlook.com,M,1965-02-12,Dallas,TX,USA,2022-08-03
CUST-1010,Margaret,Jackson,margaret.jackson75@outlook.com,F,1989-02-28,Miami,FL,USA,2023-09-19


In [0]:
 print("Bronze customers:", customers_bronze.count())
print("Silver customers:", customers_clean.count())

Bronze customers: 1027
Silver customers: 1001


In [0]:
display(
    customers_clean
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
)

customer_id,count


In [0]:
silver_base = "abfss://shopsphere@stshopsphere2026.dfs.core.windows.net/silver"

customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base}/customers")

In [0]:
customers_silver = spark.read.format("delta").load(
    f"{silver_base}/customers"
)

print("Silver customers:", customers_silver.count())
display(customers_silver.limit(10))

Silver customers: 1001


customer_id,first_name,last_name,email,gender,date_of_birth,city,state,country,signup_date
CUST-1019,Robert,Williams,robert.williams79@outlook.com,M,1987-04-29,San Diego,CA,USA,2021-10-21
CUST-1100,Mark,Hernandez,mark.hernandez35@gmail.com,M,1982-12-11,Boston,MA,USA,2023-06-20
CUST-1131,Betty,Moore,betty.moore36@icloud.com,F,1970-10-10,Houston,TX,USA,2021-07-18
CUST-1163,Lisa,Martinez,lisa.martinez93@gmail.com,F,1998-06-16,New York,NY,USA,2022-03-23
CUST-1203,Jessica,Lopez,jessica.lopez98@hotmail.com,F,1973-05-29,Chicago,IL,USA,2023-06-02
CUST-1207,Robert,Wilson,robert.wilson36@outlook.com,M,1981-03-14,Chicago,IL,USA,2022-05-31
CUST-1210,Margaret,Martin,margaret.martin18@hotmail.com,F,1986-06-28,San Antonio,TX,USA,2022-10-06
CUST-1236,Sarah,Miller,sarah.miller55@gmail.com,F,1978-05-10,San Diego,CA,USA,2021-03-08
CUST-1254,Thomas,Jackson,thomas.jackson2@icloud.com,M,1980-10-02,Miami,FL,USA,2022-03-13
CUST-1258,Linda,Taylor,linda.taylor78@icloud.com,F,1978-04-05,Denver,CO,USA,2021-04-22


In [0]:
products_bronze = spark.read.format("delta").load(
    f"{bronze_base}/products"
)

products_bronze.printSchema()
display(products_bronze.limit(20))

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)



product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-101,GAP Men's Graphic Tee - Khaki,Men,T-Shirts,Men,Khaki,S,GAP,24.45,10.36,0
PRD-102,GAP Men's Polo - Navy,Men,T-Shirts,Men,Navy,XS,GAP,21.86,8.16,0
PRD-103,GAP Men's Denim Shirt - White,Men,Shirts,Men,White,L,GAP,57.67,24.85,0
PRD-104,GAP Men's Denim Shirt - Olive,Men,Shirts,Men,Olive,M,GAP,44.65,19.9,0
PRD-105,GAP Men's Zip Hoodie - Heather Grey,Men,Hoodies,Men,Heather Grey,M,GAP,83.89,34.43,16
PRD-106,GAP Men's Fleece Hoodie - Heather Grey,Men,Hoodies,Men,Heather Grey,L,GAP,53.14,20.16,165
PRD-107,GAP Men's Crewneck Sweatshirt - Navy,Men,Sweatshirts,Men,Navy,XXL,GAP,59.11,22.4,0
PRD-108,GAP Men's Crewneck Sweatshirt - Red,Men,Sweatshirts,Men,Red,M,GAP,68.13,29.53,46
PRD-109,GAP Men's Straight Jeans - Khaki,Men,Jeans,Men,Khaki,XXL,GAP,87.46,36.28,0
PRD-110,GAP Men's Wide-Leg Jeans - White,Men,Jeans,Men,White,S,GAP,68.48,25.82,0


In [0]:
from pyspark.sql.functions import col, trim, lower

products_clean = (
    products_bronze
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("subcategory", trim(col("subcategory")))
    .withColumn("gender", trim(col("gender")))
    .withColumn("color", trim(col("color")))
    .withColumn("size", trim(col("size")))
    .withColumn("brand", trim(col("brand")))
)

In [0]:
products_clean = products_clean.filter(
    col("product_id").isNotNull() &
    (col("product_id") != "")
)

In [0]:
from pyspark.sql.functions import count

product_duplicates = (
    products_clean
    .groupBy("product_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

display(product_duplicates)

product_id,count


In [0]:
display(
    products_clean.filter(
        col("price").isNull() | (col("price") <= 0)
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-99999,GAP Discontinued Item,Men,T-Shirts,Men,Unknown,M,GAP,null,-10.0,-5


In [0]:
display(
    products_clean.filter(
        col("cost").isNull() | (col("cost") < 0)
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-99999,GAP Discontinued Item,Men,T-Shirts,Men,Unknown,M,GAP,null,-10.0,-5


In [0]:
display(
    products_clean.filter(
        col("cost") > col("price")
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity


In [0]:
display(
    products_clean.filter(
        col("stock_quantity").isNull() |
        (col("stock_quantity") < 0)
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-99999,GAP Discontinued Item,Men,T-Shirts,Men,Unknown,M,GAP,null,-10.0,-5


In [0]:
display(
    products_clean.filter(
        col("product_name").isNull() |
        (col("product_name") == "") |
        col("category").isNull() |
        (col("category") == "") |
        col("brand").isNull() |
        (col("brand") == "")
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity


In [0]:
products_clean = products_clean.filter(
    col("price").isNotNull() &
    (col("price") > 0) &
    col("cost").isNotNull() &
    (col("cost") >= 0)
)

In [0]:
display(
    products_clean.filter(
        col("price").isNull() |
        (col("price") <= 0) |
        col("cost").isNull() |
        (col("cost") < 0)
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity


In [0]:
display(
    products_clean.filter(
        col("cost") > col("price")
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity


In [0]:
display(
    products_clean.filter(
        col("stock_quantity").isNull() |
        (col("stock_quantity") < 0)
    )
)

product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity


In [0]:
from pyspark.sql.functions import col, count

# ==============================
# PRODUCTS SILVER - DQ CHECK
# ==============================

print("Total rows:", products_clean.count())
print("Total columns:", len(products_clean.columns))

# 1. Missing product IDs
print("\n1. Missing product_id:")
print(
    products_clean.filter(
        col("product_id").isNull() |
        (col("product_id") == "")
    ).count()
)

# 2. Duplicate product IDs
print("\n2. Duplicate product_id:")
print(
    products_clean
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

# 3. Missing product names
print("\n3. Missing product_name:")
print(
    products_clean.filter(
        col("product_name").isNull() |
        (col("product_name") == "")
    ).count()
)

# 4. Invalid prices
print("\n4. Invalid price:")
print(
    products_clean.filter(
        col("price").isNull() |
        (col("price") <= 0)
    ).count()
)

# 5. Invalid costs
print("\n5. Invalid cost:")
print(
    products_clean.filter(
        col("cost").isNull() |
        (col("cost") < 0)
    ).count()
)

# 6. Cost greater than price
print("\n6. Cost > Price:")
print(
    products_clean.filter(
        col("cost") > col("price")
    ).count()
)

# 7. Invalid stock
print("\n7. Invalid stock_quantity:")
print(
    products_clean.filter(
        col("stock_quantity").isNull() |
        (col("stock_quantity") < 0)
    ).count()
)

# 8. Missing category
print("\n8. Missing category:")
print(
    products_clean.filter(
        col("category").isNull() |
        (col("category") == "")
    ).count()
)

# 9. Missing brand
print("\n9. Missing brand:")
print(
    products_clean.filter(
        col("brand").isNull() |
        (col("brand") == "")
    ).count()
)

# 10. Show final schema
print("\n10. Schema:")
products_clean.printSchema()

# 11. Sample final data
print("\n11. Sample:")
display(products_clean.limit(10))

Total rows: 104
Total columns: 11

1. Missing product_id:
0

2. Duplicate product_id:
0

3. Missing product_name:
0

4. Invalid price:
0

5. Invalid cost:
0

6. Cost > Price:
0

7. Invalid stock_quantity:
0

8. Missing category:
0

9. Missing brand:
0

10. Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- color: string (nullable = true)
 |-- size: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)


11. Sample:


product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-101,GAP Men's Graphic Tee - Khaki,Men,T-Shirts,Men,Khaki,S,GAP,24.45,10.36,0
PRD-102,GAP Men's Polo - Navy,Men,T-Shirts,Men,Navy,XS,GAP,21.86,8.16,0
PRD-103,GAP Men's Denim Shirt - White,Men,Shirts,Men,White,L,GAP,57.67,24.85,0
PRD-104,GAP Men's Denim Shirt - Olive,Men,Shirts,Men,Olive,M,GAP,44.65,19.9,0
PRD-105,GAP Men's Zip Hoodie - Heather Grey,Men,Hoodies,Men,Heather Grey,M,GAP,83.89,34.43,16
PRD-106,GAP Men's Fleece Hoodie - Heather Grey,Men,Hoodies,Men,Heather Grey,L,GAP,53.14,20.16,165
PRD-107,GAP Men's Crewneck Sweatshirt - Navy,Men,Sweatshirts,Men,Navy,XXL,GAP,59.11,22.4,0
PRD-108,GAP Men's Crewneck Sweatshirt - Red,Men,Sweatshirts,Men,Red,M,GAP,68.13,29.53,46
PRD-109,GAP Men's Straight Jeans - Khaki,Men,Jeans,Men,Khaki,XXL,GAP,87.46,36.28,0
PRD-110,GAP Men's Wide-Leg Jeans - White,Men,Jeans,Men,White,S,GAP,68.48,25.82,0


In [0]:
silver_base = "abfss://shopsphere@stshopsphere2026.dfs.core.windows.net/silver"

products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base}/products")

In [0]:
products_silver = spark.read.format("delta").load(
    f"{silver_base}/products"
)

print("Products Silver count:", products_silver.count())

display(products_silver.limit(10))

Products Silver count: 104


product_id,product_name,category,subcategory,gender,color,size,brand,price,cost,stock_quantity
PRD-101,GAP Men's Graphic Tee - Khaki,Men,T-Shirts,Men,Khaki,S,GAP,24.45,10.36,0
PRD-102,GAP Men's Polo - Navy,Men,T-Shirts,Men,Navy,XS,GAP,21.86,8.16,0
PRD-103,GAP Men's Denim Shirt - White,Men,Shirts,Men,White,L,GAP,57.67,24.85,0
PRD-104,GAP Men's Denim Shirt - Olive,Men,Shirts,Men,Olive,M,GAP,44.65,19.9,0
PRD-105,GAP Men's Zip Hoodie - Heather Grey,Men,Hoodies,Men,Heather Grey,M,GAP,83.89,34.43,16
PRD-106,GAP Men's Fleece Hoodie - Heather Grey,Men,Hoodies,Men,Heather Grey,L,GAP,53.14,20.16,165
PRD-107,GAP Men's Crewneck Sweatshirt - Navy,Men,Sweatshirts,Men,Navy,XXL,GAP,59.11,22.4,0
PRD-108,GAP Men's Crewneck Sweatshirt - Red,Men,Sweatshirts,Men,Red,M,GAP,68.13,29.53,46
PRD-109,GAP Men's Straight Jeans - Khaki,Men,Jeans,Men,Khaki,XXL,GAP,87.46,36.28,0
PRD-110,GAP Men's Wide-Leg Jeans - White,Men,Jeans,Men,White,S,GAP,68.48,25.82,0


In [0]:
orders_bronze = spark.read.format("delta").load(
    f"{bronze_base}/orders"
)

orders_bronze.printSchema()
display(orders_bronze.limit(20))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_amount: double (nullable = true)



order_id,customer_id,product_id,order_date,quantity,unit_price,discount,shipping_cost,payment_method,order_status,total_amount
ORD-100001,CUST-1562,PRD-160,2022-06-22T14:07:00.000Z,2,46.74,0.1,0.0,Credit Card,Completed,84.13
ORD-100002,CUST-1733,PRD-104,2022-06-14T10:57:00.000Z,3,44.65,0.15,0.0,GAP Gift Card,Completed,113.86
ORD-100003,CUST-1945,PRD-169,2023-10-11T23:31:00.000Z,3,28.93,0.3,0.0,Klarna,Completed,60.75
ORD-100004,CUST-1387,PRD-192,2024-01-31T23:46:00.000Z,4,31.69,0.1,0.0,Apple Pay,Completed,114.08
ORD-100005,CUST-1115,PRD-112,2022-12-15T23:37:00.000Z,1,59.65,0.1,0.0,Apple Pay,Processing,53.69
ORD-100006,CUST-1002,PRD-140,2022-03-05T18:34:00.000Z,4,112.07,0.15,0.0,Apple Pay,Completed,381.04
ORD-100007,CUST-1617,PRD-176,2022-08-16T03:25:00.000Z,-2,20.01,0.5,5.99,PayPal,Cancelled,-14.02
ORD-100008,CUST-1151,PRD-177,2022-10-18T04:47:00.000Z,1,23.9,0.15,5.99,Credit Card,Completed,26.3
ORD-100009,CUST-1724,PRD-174,2023-05-05T20:53:00.000Z,3,24.26,0.2,0.0,Credit Card,Shipped,58.22
ORD-100010,CUST-1734,PRD-127,2024-05-18T23:59:00.000Z,1,36.14,0.5,0.0,GAP Gift Card,Processing,18.07


In [0]:
from pyspark.sql.functions import col, trim, to_date

orders_clean = (
    orders_bronze
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn("order_status", trim(col("order_status")))
)

In [0]:
orders_clean.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
orders_clean = orders_clean.filter(
    col("order_id").isNotNull() &
    (col("order_id") != "") &
    col("customer_id").isNotNull() &
    (col("customer_id") != "") &
    col("product_id").isNotNull() &
    (col("product_id") != "")
)

In [0]:
display(
    orders_clean
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

order_id,count
ORD-100034,2
ORD-100036,2
ORD-100040,2
ORD-100097,2
ORD-100108,2
ORD-100120,2
ORD-100252,2
ORD-100322,2
ORD-100440,2
ORD-100511,2


In [0]:
orders_clean = orders_clean.dropDuplicates(["order_id"])

In [0]:
print("Invalid quantity:",
      orders_clean.filter(
          col("quantity").isNull() | (col("quantity") <= 0)
      ).count())

print("Invalid unit_price:",
      orders_clean.filter(
          col("unit_price").isNull() | (col("unit_price") <= 0)
      ).count())

print("Invalid discount:",
      orders_clean.filter(
          col("discount").isNull() |
          (col("discount") < 0) |
          (col("discount") > 1)
      ).count())

print("Invalid shipping_cost:",
      orders_clean.filter(
          col("shipping_cost").isNull() |
          (col("shipping_cost") < 0)
      ).count())

print("Invalid total_amount:",
      orders_clean.filter(
          col("total_amount").isNull() |
          (col("total_amount") < 0)
      ).count())

Invalid quantity: 109
Invalid unit_price: 0
Invalid discount: 93
Invalid shipping_cost: 0
Invalid total_amount: 200


In [0]:
print("Missing order_date:",
      orders_clean.filter(
          col("order_date").isNull()
      ).count())

Missing order_date: 0


In [0]:
orders_clean.select(
    "order_date"
).summary("min", "max").show()

+-------+
|summary|
+-------+
|    min|
|    max|
+-------+



In [0]:
customers_silver = spark.read.format("delta").load(
    f"{silver_base}/customers"
)

products_silver = spark.read.format("delta").load(
    f"{silver_base}/products"
)

In [0]:
invalid_customers = (
    orders_clean
    .join(
        customers_silver.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print("Orders with invalid customer_id:", invalid_customers.count())

Orders with invalid customer_id: 0


In [0]:
invalid_products = (
    orders_clean
    .join(
        products_silver.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print("Orders with invalid product_id:", invalid_products.count())

Orders with invalid product_id: 256


In [0]:
orders_clean = orders_clean.filter(
    col("quantity").isNotNull() &
    (col("quantity") > 0) &
    
    col("discount").isNotNull() &
    (col("discount") >= 0) &
    (col("discount") <= 1) &
    
    col("total_amount").isNotNull() &
    (col("total_amount") >= 0)
)

In [0]:
print("Invalid quantity:",
      orders_clean.filter(
          col("quantity").isNull() | (col("quantity") <= 0)
      ).count())

print("Invalid discount:",
      orders_clean.filter(
          col("discount").isNull() |
          (col("discount") < 0) |
          (col("discount") > 1)
      ).count())

print("Invalid total_amount:",
      orders_clean.filter(
          col("total_amount").isNull() |
          (col("total_amount") < 0)
      ).count())

Invalid quantity: 0
Invalid discount: 0
Invalid total_amount: 0


In [0]:
orders_clean = (
    orders_clean
    .join(
        products_silver.select("product_id"),
        on="product_id",
        how="inner"
    )
)

In [0]:
invalid_products = (
    orders_clean
    .join(
        products_silver.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print("Orders with invalid product_id:", invalid_products.count())

Orders with invalid product_id: 0


In [0]:
invalid_customers = (
    orders_clean
    .join(
        customers_silver.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print("Orders with invalid customer_id:", invalid_customers.count())

Orders with invalid customer_id: 0


In [0]:
print("Missing order_date:",
      orders_clean.filter(col("order_date").isNull()).count())

print("Final Orders Silver candidate:", orders_clean.count())

Missing order_date: 0
Final Orders Silver candidate: 9339


In [0]:
print("===== ORDERS SILVER FINAL DQ CHECK =====")

print("Total rows:", orders_clean.count())

print("Missing order_id:",
      orders_clean.filter(
          col("order_id").isNull() | (col("order_id") == "")
      ).count())

print("Duplicate order_id:",
      orders_clean.groupBy("order_id")
      .count()
      .filter(col("count") > 1)
      .count())

print("Missing customer_id:",
      orders_clean.filter(
          col("customer_id").isNull() | (col("customer_id") == "")
      ).count())

print("Missing product_id:",
      orders_clean.filter(
          col("product_id").isNull() | (col("product_id") == "")
      ).count())

print("Invalid quantity:",
      orders_clean.filter(
          col("quantity").isNull() | (col("quantity") <= 0)
      ).count())

print("Invalid unit_price:",
      orders_clean.filter(
          col("unit_price").isNull() | (col("unit_price") <= 0)
      ).count())

print("Invalid discount:",
      orders_clean.filter(
          col("discount").isNull() |
          (col("discount") < 0) |
          (col("discount") > 1)
      ).count())

print("Invalid shipping_cost:",
      orders_clean.filter(
          col("shipping_cost").isNull() |
          (col("shipping_cost") < 0)
      ).count())

print("Invalid total_amount:",
      orders_clean.filter(
          col("total_amount").isNull() |
          (col("total_amount") < 0)
      ).count())

print("Missing order_date:",
      orders_clean.filter(
          col("order_date").isNull()
      ).count())

===== ORDERS SILVER FINAL DQ CHECK =====
Total rows: 9339
Missing order_id: 0
Duplicate order_id: 0
Missing customer_id: 0
Missing product_id: 0
Invalid quantity: 0
Invalid unit_price: 0
Invalid discount: 0
Invalid shipping_cost: 0
Invalid total_amount: 0
Missing order_date: 0


In [0]:
orders_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base}/orders")

In [0]:
orders_silver = spark.read.format("delta").load(
    f"{silver_base}/orders"
)

print("Orders Silver count:", orders_silver.count())

display(orders_silver.limit(10))

Orders Silver count: 9339


product_id,order_id,customer_id,order_date,quantity,unit_price,discount,shipping_cost,payment_method,order_status,total_amount
PRD-194,ORD-100012,CUST-1541,2022-10-17T09:44:00.000Z,4,42.02,0.15,0.0,Klarna,Completed,142.87
PRD-120,ORD-100032,CUST-1731,2023-11-05T04:09:00.000Z,3,38.57,0.4,0.0,Apple Pay,Completed,69.43
PRD-112,ORD-100035,CUST-1968,2022-06-19T18:03:00.000Z,3,59.65,0.2,0.0,Credit Card,Processing,143.16
PRD-199,ORD-100039,CUST-1872,2024-05-28T03:18:00.000Z,3,21.93,0.0,0.0,Klarna,Processing,65.79
PRD-166,ORD-100052,CUST-1063,2023-03-14T20:03:00.000Z,3,49.85,0.15,0.0,GAP Gift Card,Completed,127.12
PRD-193,ORD-100057,CUST-1309,2023-05-01T06:34:00.000Z,1,31.54,0.2,5.99,Apple Pay,Returned,31.22
PRD-150,ORD-100070,CUST-1387,2022-01-22T03:52:00.000Z,1,14.74,0.5,0.0,PayPal,Completed,7.37
PRD-138,ORD-100170,CUST-1536,2023-10-01T17:42:00.000Z,4,59.71,0.5,0.0,Apple Pay,Completed,119.42
PRD-190,ORD-100185,CUST-1760,2023-07-14T06:30:00.000Z,3,38.12,0.3,0.0,GAP Gift Card,Shipped,80.05
PRD-133,ORD-100220,CUST-1676,2024-03-12T20:51:00.000Z,1,60.6,0.3,0.0,Apple Pay,Completed,42.42


In [0]:
print("=" * 50)
print("SHOPSPHERE SILVER TRANSFORMATION")
print("=" * 50)

print(f"Customers Silver: {customers_clean.count()}")
print(f"Products Silver:  {products_clean.count()}")
print(f"Orders Silver:    {orders_clean.count()}")

print("Silver transformations completed successfully.")

SHOPSPHERE SILVER TRANSFORMATION
Customers Silver: 1001
Products Silver:  104
Orders Silver:    9339
Silver transformations completed successfully.
